# **Практическая работа. Геоанализ экологических факторов городской среды с применением методов пространственной кластеризации**




















## **Цель работы**



Освоение методов пространственного анализа и машинного обучения для оценки экологической обстановки городской территории с использованием гексагональной сетки H3, с последующим применением различных алгоритмов кластеризации и классификации.

## **Задачи**



1. Освоить инструменты загрузки и агрегации экологических геоданных с использованием API OpenStreetMap
2. Реализовать анализ пространственного распределения экологических факторов с помощью гексагональной сетки H3
3. Применить и сравнить различные алгоритмы кластеризации для выявления однородных экологических зон
4. Обучить модели классификации для прогнозирования экологического состояния новых территорий

## **Теоретическая часть**



Современный геоэкологический анализ требует комплексного подхода к обработке пространственно-распределенных данных. Применение методов машинного обучения, в частности алгоритмов кластеризации (K-means, агломеративной, спектральной, DBSCAN, HDBSCAN), позволяет выявлять неявные закономерности в распределении экологических факторов городской среды и определять территории со схожими экологическими характеристиками.

## **Этапы работы**



### **1. Определение области исследования и подготовка данных**


- Выберите городскую территорию для анализа экологической обстановки
- Используя API OpenStreetMap, загрузите данные следующих категорий:
  - Источники загрязнения (промышленные предприятия, мусоропереработка, ТЭЦ)
  - Зеленые насаждения (парки, скверы, лесопарковые зоны)
  - Водные объекты (реки, водоемы)
  - Автомагистрали (категории дорог с интенсивным движением)
  - Административные районы города

In [ ]:
# Этап 1. Определение территории исследования и загрузка данных OpenStreetMap

import sys
import subprocess
import importlib.util
import warnings

warnings.filterwarnings("ignore")

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "geopandas": "geopandas",
    "shapely": "shapely",
    "folium": "folium",
    "leafmap": "leafmap",
    "h3": "h3",
    "osmnx": "osmnx",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
}

missing_packages = [pip_name for import_name, pip_name in required_packages.items() if importlib.util.find_spec(import_name) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import leafmap
import h3
import osmnx as ox
import matplotlib.pyplot as plt
import seaborn as sns

from shapely.geometry import Point, Polygon, LineString, box
from IPython.display import display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Для анализа выбрана юго-восточная часть Москвы.
# В этой зоне сочетаются жилые кварталы, крупные дороги, промышленные объекты,
# парки и водные объекты, поэтому территория хорошо подходит для учебного геоэкологического анализа.
study_area_name = "Юго-восточная часть Москвы"
roi_polygon = box(37.62, 55.64, 37.82, 55.78)
roi_gdf = gpd.GeoDataFrame({"name": [study_area_name]}, geometry=[roi_polygon], crs="EPSG:4326")

data_sources = pd.DataFrame([
    ["Источники загрязнения", "landuse=industrial; man_made=works; amenity=waste_transfer_station", "Промышленные зоны, предприятия, объекты обращения с отходами"],
    ["Зеленые насаждения", "leisure=park; landuse=forest/grass; natural=wood/scrub", "Парки, лесопарки, зеленые территории"],
    ["Водные объекты", "natural=water; waterway=river/canal/stream", "Реки, каналы, водоемы"],
    ["Автомагистрали", "highway=motorway/trunk/primary/secondary", "Дороги с высокой транспортной нагрузкой"],
    ["Административные районы", "boundary=administrative", "Контекстные границы для интерпретации результатов"],
], columns=["Категория", "Теги OpenStreetMap", "Как используется в анализе"])

display(pd.DataFrame({
    "Территория": [study_area_name],
    "Смысл выбора": ["На территории есть контраст между транспортной нагрузкой, промышленными зонами, парками и водными объектами."],
}))
display(data_sources)

def ox_features_from_polygon(polygon, tags):
    if hasattr(ox, "features_from_polygon"):
        return ox.features_from_polygon(polygon, tags)
    return ox.geometries_from_polygon(polygon, tags)

def ensure_crs(gdf):
    if gdf is None or len(gdf) == 0:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    gdf = gdf[gdf.geometry.notna()].copy()
    return gdf.set_crs("EPSG:4326", allow_override=True)

def clip_to_roi(gdf):
    gdf = ensure_crs(gdf)
    if len(gdf) == 0:
        return gdf
    return gpd.clip(gdf, roi_gdf)

def make_demo_points(n, label):
    minx, miny, maxx, maxy = roi_polygon.bounds
    pts = []
    while len(pts) < n:
        p = Point(np.random.uniform(minx, maxx), np.random.uniform(miny, maxy))
        if roi_polygon.contains(p):
            pts.append(p)
    return gpd.GeoDataFrame({"type": [label] * n}, geometry=pts, crs="EPSG:4326")

def make_demo_polygons(n, label, size=0.008):
    points = make_demo_points(n, label)
    polygons = points.copy()
    polygons["geometry"] = polygons.geometry.buffer(size)
    polygons = polygons.set_crs("EPSG:4326", allow_override=True)
    return gpd.clip(polygons, roi_gdf)

def make_demo_lines(n, label):
    minx, miny, maxx, maxy = roi_polygon.bounds
    lines = []
    for _ in range(n):
        x1 = np.random.uniform(minx, maxx)
        y1 = np.random.uniform(miny, maxy)
        x2 = min(maxx, max(minx, x1 + np.random.uniform(-0.04, 0.04)))
        y2 = min(maxy, max(miny, y1 + np.random.uniform(-0.02, 0.02)))
        lines.append(LineString([(x1, y1), (x2, y2)]))
    return gpd.GeoDataFrame({"type": [label] * n}, geometry=lines, crs="EPSG:4326")

osm_tags = {
    "pollution": {"landuse": "industrial", "man_made": "works", "amenity": ["waste_transfer_station", "recycling"]},
    "green": {"leisure": ["park", "garden"], "landuse": ["forest", "grass", "recreation_ground"], "natural": ["wood", "scrub"]},
    "water": {"natural": "water", "waterway": ["river", "canal", "stream"]},
    "roads": {"highway": ["motorway", "trunk", "primary", "secondary"]},
    "districts": {"boundary": "administrative"},
}

layers = {}
used_demo_data = False

try:
    for layer_name, tags in osm_tags.items():
        layers[layer_name] = clip_to_roi(ox_features_from_polygon(roi_polygon, tags))
except Exception as exc:
    used_demo_data = True
    print("Данные OSM не удалось загрузить. Для демонстрации методики используются синтетические объекты.")
    print(type(exc).__name__, str(exc)[:300])
    layers = {
        "pollution": make_demo_points(35, "pollution"),
        "green": make_demo_polygons(22, "green", size=0.007),
        "water": make_demo_lines(8, "water"),
        "roads": make_demo_lines(45, "road"),
        "districts": roi_gdf.copy(),
    }

for name, gdf in layers.items():
    print(f"{name}: {len(gdf)} объектов")

m_data = folium.Map(location=[55.71, 37.72], zoom_start=12, tiles="CartoDB positron")
folium.GeoJson(roi_gdf, name="Область исследования", style_function=lambda x: {"color": "black", "weight": 2, "fillOpacity": 0}).add_to(m_data)

colors = {"pollution": "red", "green": "green", "water": "blue", "roads": "orange", "districts": "gray"}
for name, gdf in layers.items():
    if len(gdf) == 0:
        continue
    folium.GeoJson(
        gdf,
        name=name,
        style_function=lambda x, color=colors.get(name, "gray"): {"color": color, "weight": 2, "fillOpacity": 0.25},
    ).add_to(m_data)

folium.LayerControl().add_to(m_data)
m_data


### **2. Агрегация данных с использованием гексагональной сетки H3**


- Сгенерируйте гексагональную сетку H3 оптимального разрешения для выбранной территории
- Для каждой ячейки H3 рассчитайте:
  - Количество и плотность источников загрязнения в радиусе 1600м
  - Площадь и процент покрытия зелеными насаждениями в радиусе 800м
  - Протяженность водных объектов в радиусе 1600м
  - Плотность автомагистралей в радиусе 1600м
- Выполните нормализацию полученных показателей
- Сформируйте интегральный индекс экологического благополучия территории


In [ ]:
# Этап 2. H3-сетка, агрегация факторов и интегральный экологический индекс

from sklearn.preprocessing import MinMaxScaler

h3_resolution = 8
print("Выбранное разрешение H3:", h3_resolution)
print("Обоснование: resolution 8 дает ячейки городского квартального масштаба, удобные для оценки факторов в радиусах 800-1600 м.")

def h3_to_cell(lat, lon, resolution):
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, resolution)
    return h3.geo_to_h3(lat, lon, resolution)

def h3_boundary(cell_id):
    if hasattr(h3, "cell_to_boundary"):
        boundary = h3.cell_to_boundary(cell_id)
    else:
        boundary = h3.h3_to_geo_boundary(cell_id)
    return Polygon([(lng, lat) for lat, lng in boundary])

def build_h3_grid(polygon, resolution, step_deg=0.002):
    minx, miny, maxx, maxy = polygon.bounds
    cells = set()
    for lon in np.arange(minx, maxx + step_deg, step_deg):
        for lat in np.arange(miny, maxy + step_deg, step_deg):
            point = Point(lon, lat)
            if polygon.contains(point):
                cells.add(h3_to_cell(lat, lon, resolution))
    grid = gpd.GeoDataFrame({"h3_cell": list(cells)}, geometry=[h3_boundary(c) for c in cells], crs="EPSG:4326")
    grid = grid[grid.intersects(polygon)].copy().reset_index(drop=True)
    grid["cell_id"] = np.arange(len(grid))
    return grid

grid = build_h3_grid(roi_polygon, h3_resolution)
print("Количество H3-ячеек:", len(grid))

grid_m = grid.to_crs(epsg=3857)
grid_m["centroid"] = grid_m.geometry.centroid
centroids = gpd.GeoDataFrame(grid_m[["cell_id"]].copy(), geometry=grid_m["centroid"], crs="EPSG:3857")

pollution_m = layers["pollution"].to_crs(epsg=3857)
green_m = layers["green"].to_crs(epsg=3857)
water_m = layers["water"].to_crs(epsg=3857)
roads_m = layers["roads"].to_crs(epsg=3857)

def count_objects_within(gdf, radius_m):
    if len(gdf) == 0:
        return np.zeros(len(centroids))
    geoms = list(gdf.geometry)
    return np.array([sum(center.distance(geom.representative_point()) <= radius_m for geom in geoms) for center in centroids.geometry])

def area_intersection_within(gdf, radius_m):
    if len(gdf) == 0:
        return np.zeros(len(centroids))
    values = []
    for center in centroids.geometry:
        buffer = center.buffer(radius_m)
        values.append(gdf[gdf.intersects(buffer)].intersection(buffer).area.sum())
    return np.array(values)

def length_intersection_within(gdf, radius_m):
    if len(gdf) == 0:
        return np.zeros(len(centroids))
    values = []
    for center in centroids.geometry:
        buffer = center.buffer(radius_m)
        values.append(gdf[gdf.intersects(buffer)].intersection(buffer).length.sum())
    return np.array(values)

features = grid.copy()
features["pollution_count_1600m"] = count_objects_within(pollution_m, 1600)
features["pollution_density_1600m"] = features["pollution_count_1600m"] / (np.pi * 1.6 ** 2)
features["green_area_800m_m2"] = area_intersection_within(green_m, 800)
features["green_share_800m"] = features["green_area_800m_m2"] / (np.pi * 800 ** 2)
features["water_length_1600m_m"] = length_intersection_within(water_m, 1600)
features["road_length_1600m_m"] = length_intersection_within(roads_m, 1600)
features["road_density_1600m"] = features["road_length_1600m_m"] / (np.pi * 1.6 ** 2)

factor_cols = [
    "pollution_density_1600m",
    "green_share_800m",
    "water_length_1600m_m",
    "road_density_1600m",
]

for col in factor_cols:
    lo, hi = features[col].quantile([0.01, 0.99])
    features[col] = features[col].clip(lo, hi)

scaled = pd.DataFrame(MinMaxScaler().fit_transform(features[factor_cols]), columns=[f"{c}_norm" for c in factor_cols], index=features.index)
features = pd.concat([features, scaled], axis=1)

# Логика индекса: зеленые зоны и вода улучшают экологическое состояние,
# источники загрязнения и автомагистрали ухудшают его.
features["eco_index"] = (
    0.35 * features["green_share_800m_norm"] +
    0.20 * features["water_length_1600m_m_norm"] +
    0.25 * (1 - features["pollution_density_1600m_norm"]) +
    0.20 * (1 - features["road_density_1600m_norm"])
)
features["eco_class"] = pd.qcut(features["eco_index"], 4, labels=["низкое", "умеренное", "хорошее", "высокое"], duplicates="drop")

display(features[factor_cols + [f"{c}_norm" for c in factor_cols] + ["eco_index", "eco_class"]].describe().T)

fig, ax = plt.subplots(figsize=(8, 8))
features.plot(column="eco_index", cmap="YlGn", legend=True, edgecolor="white", linewidth=0.25, ax=ax)
ax.set_title("Интегральный индекс экологического благополучия")
ax.set_axis_off()
plt.show()

features.head()


### **3. Определение оптимального числа кластеров**


- Постройте график метода локтя (Elbow method) для определения оптимального числа кластеров
- Оцените качество кластеризации с помощью внутренних метрик кластеризации
- Определите оптимальное число кластеров на основе комбинации различных метрик

In [ ]:
# Этап 3. Определение оптимального числа кластеров

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

cluster_features = [
    "pollution_density_1600m_norm",
    "green_share_800m_norm",
    "water_length_1600m_m_norm",
    "road_density_1600m_norm",
    "eco_index",
]

X = features[cluster_features].fillna(0).copy()
X_scaled = StandardScaler().fit_transform(X)

metrics = []
k_values = range(2, min(10, len(features) - 1))

for k in k_values:
    labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit_predict(X_scaled)
    metrics.append({
        "k": k,
        "inertia": KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit(X_scaled).inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
    })

metrics_df = pd.DataFrame(metrics)
display(metrics_df)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(metrics_df["k"], metrics_df["inertia"], marker="o")
axes[0].set_title("Метод локтя")
axes[0].set_xlabel("Число кластеров")
axes[0].set_ylabel("Inertia")

axes[1].plot(metrics_df["k"], metrics_df["silhouette"], marker="o", color="green")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("Число кластеров")

axes[2].plot(metrics_df["k"], metrics_df["davies_bouldin"], marker="o", color="red")
axes[2].set_title("Davies-Bouldin index")
axes[2].set_xlabel("Число кластеров")
plt.tight_layout()
plt.show()

best_k = int(metrics_df.sort_values(["silhouette", "calinski_harabasz"], ascending=False).iloc[0]["k"])
print("Оптимальное число кластеров по совокупности метрик:", best_k)


### **4. Сравнительный анализ алгоритмов кластеризации**


- Реализуйте и сравните следующие алгоритмы кластеризации:
  - K-means
  - Агломеративная кластеризация
  - Спектральная кластеризация
  - DBSCAN (с оптимальным значением eps)
  - HDBSCAN
- Для каждого алгоритма оцените:
  - Качество кластеризации по основным метрикам
  - Распределение точек по кластерам
  - Характерные особенности выявленных кластеров

In [ ]:
# Этап 4. Сравнение алгоритмов кластеризации

from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors

def evaluate_clustering(name, labels):
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    noise_share = float(np.mean(labels == -1))
    if n_clusters >= 2 and mask.sum() > n_clusters:
        sil = silhouette_score(X_scaled[mask], labels[mask])
        ch = calinski_harabasz_score(X_scaled[mask], labels[mask])
        db = davies_bouldin_score(X_scaled[mask], labels[mask])
    else:
        sil, ch, db = np.nan, np.nan, np.nan
    return {
        "algorithm": name,
        "n_clusters": n_clusters,
        "noise_share": noise_share,
        "silhouette": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db,
    }

clustering_results = {}

clustering_results["KMeans"] = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20).fit_predict(X_scaled)
clustering_results["Agglomerative"] = AgglomerativeClustering(n_clusters=best_k).fit_predict(X_scaled)
clustering_results["Spectral"] = SpectralClustering(n_clusters=best_k, affinity="nearest_neighbors", random_state=RANDOM_STATE).fit_predict(X_scaled)

# Подбор eps для DBSCAN через расстояние до 5-го соседа.
neighbors = NearestNeighbors(n_neighbors=min(5, len(X_scaled))).fit(X_scaled)
distances, _ = neighbors.kneighbors(X_scaled)
eps_value = float(np.percentile(distances[:, -1], 85))
clustering_results["DBSCAN"] = DBSCAN(eps=eps_value, min_samples=5).fit_predict(X_scaled)
print("Подобранный eps для DBSCAN:", round(eps_value, 3))

try:
    import hdbscan
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "hdbscan"])
        import hdbscan
    except Exception as exc:
        hdbscan = None
        print("HDBSCAN недоступен в текущей среде, поэтому он будет пропущен.")
        print(type(exc).__name__, str(exc)[:200])

if hdbscan is not None:
    clustering_results["HDBSCAN"] = hdbscan.HDBSCAN(min_cluster_size=8, min_samples=4).fit_predict(X_scaled)

comparison = pd.DataFrame([evaluate_clustering(name, labels) for name, labels in clustering_results.items()])
display(comparison.sort_values(["silhouette", "calinski_harabasz"], ascending=False))

best_algorithm = comparison.dropna(subset=["silhouette"]).sort_values(["silhouette", "calinski_harabasz"], ascending=False).iloc[0]["algorithm"]
best_labels = clustering_results[best_algorithm]
features["best_cluster"] = best_labels

print("Лучший алгоритм кластеризации:", best_algorithm)

cluster_profiles = features.groupby("best_cluster")[cluster_features + ["eco_index"]].mean().round(3)
display(cluster_profiles)

fig, ax = plt.subplots(figsize=(8, 8))
features.plot(column="best_cluster", categorical=True, legend=True, edgecolor="white", linewidth=0.25, ax=ax)
ax.set_title(f"Кластеры экологических зон: {best_algorithm}")
ax.set_axis_off()
plt.show()


### **5. Визуализация и интерпретация результатов**


- Визуализируйте результаты кластеризации на карте с использованием leafmap
- Постройте тепловые карты средних значений экологических факторов для каждого кластера
- Выполните снижение размерности с помощью PCA и визуализируйте кластеры в двумерном пространстве
- Проанализируйте профили кластеров и составьте их экологические характеристики
- Разработайте рекомендации по улучшению экологической обстановки для каждого типа территории

In [ ]:
# Этап 5. Визуализация, PCA и интерпретация экологических зон

from sklearn.decomposition import PCA

m_clusters = leafmap.Map(center=[55.71, 37.72], zoom=12)
m_clusters.add_gdf(features[["cell_id", "eco_index", "eco_class", "best_cluster", "geometry"]], layer_name="Экологические кластеры")
m_clusters.add_gdf(roi_gdf, layer_name="Область исследования")
m_clusters

heatmap_data = features.groupby("best_cluster")[cluster_features].mean()
plt.figure(figsize=(10, 5))
sns.heatmap(heatmap_data, cmap="RdYlGn", annot=True, fmt=".2f")
plt.title("Средние значения экологических факторов по кластерам")
plt.show()

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_result = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame({
    "PC1": pca_result[:, 0],
    "PC2": pca_result[:, 1],
    "cluster": features["best_cluster"].astype(str),
    "eco_index": features["eco_index"],
})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="cluster", size="eco_index", palette="tab10", sizes=(30, 120))
plt.title("PCA-визуализация экологических кластеров")
plt.show()

cluster_descriptions = []
for cluster_id, row in cluster_profiles.iterrows():
    if cluster_id == -1:
        name = "Шумовые или переходные территории"
        recommendation = "Провести дополнительную проверку данных и точечный мониторинг."
    elif row["eco_index"] >= cluster_profiles["eco_index"].quantile(0.67):
        name = "Благоприятные зеленые территории"
        recommendation = "Сохранять зеленый каркас, ограничивать уплотнение и поддерживать рекреационные функции."
    elif row["pollution_density_1600m_norm"] >= cluster_profiles["pollution_density_1600m_norm"].quantile(0.67) or row["road_density_1600m_norm"] >= cluster_profiles["road_density_1600m_norm"].quantile(0.67):
        name = "Зоны экологической нагрузки"
        recommendation = "Усилить озеленение, шумозащиту, контроль выбросов и ограничение транзитного транспорта."
    else:
        name = "Умеренно благоприятные смешанные территории"
        recommendation = "Развивать локальное озеленение, пешеходные связи и мониторинг качества воздуха."
    cluster_descriptions.append({
        "Кластер": cluster_id,
        "Характеристика": name,
        "Средний экологический индекс": round(row["eco_index"], 3),
        "Рекомендации": recommendation,
    })

cluster_descriptions_df = pd.DataFrame(cluster_descriptions)
display(cluster_descriptions_df)


### **6. Разработка моделей классификации**


- На основе результатов лучшего алгоритма кластеризации подготовьте данные для обучения классификаторов
- Обучите и сравните различные модели классификации:
  - Логистическая регрессия
  - Дерево решений
  - Случайный лес
  - Градиентный бустинг
  - SVM
  - K-ближайших соседей
- Оцените качество моделей с использованием кросс-валидации
- Выберите оптимальную модель и сохраните её для дальнейшего использования

In [ ]:
# Этап 6. Классификация экологических зон

from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import pickle

classification_df = features[features["best_cluster"] != -1].copy()
X_cls = classification_df[cluster_features].fillna(0)
y_cls = classification_df["best_cluster"].astype(int)

if y_cls.nunique() < 2:
    print("Для классификации недостаточно кластеров. Используется классификация по квартилям экологического индекса.")
    y_cls = pd.qcut(classification_df["eco_index"], 3, labels=False, duplicates="drop").astype(int)

min_class_count = y_cls.value_counts().min()
cv_folds = max(2, min(5, int(min_class_count)))
cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)

classifiers = {
    "Логистическая регрессия": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))]),
    "Дерево решений": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE),
    "Случайный лес": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE),
    "Градиентный бустинг": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "SVM": Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE))]),
    "K-ближайших соседей": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=min(7, len(X_cls))))]),
}

scores = []
for name, model in classifiers.items():
    cv_result = cross_validate(model, X_cls, y_cls, cv=cv, scoring=["accuracy", "f1_macro"], error_score=np.nan)
    scores.append({
        "Модель": name,
        "accuracy_mean": np.nanmean(cv_result["test_accuracy"]),
        "f1_macro_mean": np.nanmean(cv_result["test_f1_macro"]),
        "f1_macro_std": np.nanstd(cv_result["test_f1_macro"]),
    })

scores_df = pd.DataFrame(scores).sort_values("f1_macro_mean", ascending=False)
display(scores_df)

best_classifier_name = scores_df.iloc[0]["Модель"]
best_classifier = classifiers[best_classifier_name]
best_classifier.fit(X_cls, y_cls)

with open("best_ecology_classifier.pkl", "wb") as f:
    pickle.dump(best_classifier, f)

print("Лучшая модель классификации:", best_classifier_name)
print("Модель сохранена в файл best_ecology_classifier.pkl")

print("Выводы:")
print("1. Территория разделена на экологические зоны по сочетанию загрязняющих факторов, зеленых насаждений, воды и дорог.")
print("2. Наиболее благоприятные зоны имеют высокий зеленый компонент и меньшую транспортно-промышленную нагрузку.")
print("3. Зоны с низким экологическим индексом требуют приоритетных мер: озеленения, снижения транспортной нагрузки и контроля источников загрязнения.")
print("4. Классификатор можно использовать для быстрой оценки новых городских участков по рассчитанным пространственным признакам.")


## Выводы

В работе выполнен геоанализ экологических факторов городской среды на примере юго-восточной части Москвы. Территория была покрыта гексагональной сеткой H3, после чего для каждой ячейки рассчитаны показатели загрязняющей нагрузки, зеленого покрытия, близости водных объектов и плотности автомагистралей.

На основе нормализованных факторов сформирован интегральный индекс экологического благополучия. Положительный вклад в индекс вносят зеленые зоны и водные объекты, отрицательный вклад — источники загрязнения и крупные дороги.

Для выявления экологически однородных зон были применены разные алгоритмы кластеризации: K-means, агломеративная кластеризация, спектральная кластеризация, DBSCAN и, при наличии библиотеки, HDBSCAN. Лучший алгоритм выбран по внутренним метрикам качества кластеризации.

Дополнительно построены модели классификации, которые позволяют прогнозировать тип экологической зоны для новых территорий по набору пространственных признаков. Практически такой подход можно использовать для предварительного экологического зонирования, выбора территорий для озеленения и определения участков, где требуется дополнительный мониторинг.

## Ограничения исследования

Результаты следует рассматривать как учебную и предварительную оценку. Данные OpenStreetMap могут быть неполными или не полностью актуальными. В работе не учитывались реальные измерения качества воздуха, уровень шума, метеоусловия, плотность населения и фактические промышленные выбросы. Для практического применения модель нужно дополнить официальными экологическими измерениями и полевыми данными.


## **Требования к отчету (Структуре блокнота)**



1. Описание выбранной территории и источников данных
2. Методика расчета экологических показателей с обоснованием выбора буферных зон и весовых коэффициентов
3. Сравнительный анализ результатов кластеризации с обоснованием выбора оптимального алгоритма
4. Карты распределения экологических факторов и результатов кластеризации
5. Подробная характеристика выявленных экологических зон (кластеров) с рекомендациями по их развитию
6. Анализ эффективности разработанных моделей классификации
7. Выводы об экологическом состоянии исследуемой территории и возможностях практического применения полученных результатов